## 1. Carga de datos
Cargamos los datos y creamos un único DataFrame con todos los proyectos (2015-2025)

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore',category=UserWarning,module='openpyxl')
pd.set_option('display.float_format','{:,.2f}'.format)

dataframes = []

for año in range(2015,2026):
    archivo = f'../data/raw/cdti/cdti_{año}.xlsx'
    df = pd.read_excel(archivo,dtype={'Código Postal':str})
    dataframes.append(df)

In [2]:
df_cdti = pd.concat(dataframes,ignore_index=True)

In [3]:
df_cdti.head()

,Razón Social,NIF Razón Social,Tipo Entidad,PYME,Título Proyecto,Fecha Aprobación / Resolución,CCAA,Provincia,Localidad,Código Postal,Tipo de Ayuda,Instrumento Financiero,Área Sectorial,CNAE,Origen de Fondos,Presupuesto,Aportación CDTI
0,2 PI R INGENIEROS 1 SL,B-60986924,EMPRESA,S,SISTEMA MODULAR ADAPTATIVO PARA EL TRATAMIENTO...,05/03/2015,Cataluña,BARCELONA,BARCELONA,08029,Ayudas Parcialmente Reembolsables,Proyectos CDTI de I+D,Prevención de la Contaminación,"Actividades profesionales, científicas y técnicas",Programa Operativo Plurirregional de España,"404.826,00 €","344.102,10 €"
1,23 DE SETEMBRE SOCIEDAD ANONIMA,A-43654151,EMPRESA,N,HIGIACOMPASS,05/03/2015,Cataluña,TARRAGONA,TARRAGONA,43003,Ayudas Parcialmente Reembolsables,Proyectos CDTI de I+D,"Aplicaciones , servicios y contenidos sectoriales","Actividades profesionales, científicas y técnicas",Programa Operativo Plurirregional de España,"638.497,00 €","399.507,57 €"
2,2IO INGENIERIA AVANZADA SL,B-87107272,EMPRESA,S,PROYECTO DE CONECTIVIDAD SMARTCOM,28/05/2015,Comunidad de Madrid,MADRID,ARGANDA DEL REY,28500,Ayudas Parcialmente Reembolsables,"Líneas de Innovación (LIC, LICa y LIEE)","Equipos, Sistemas y Servicios de Telecomunica...",Industria manufacturera,Centro de Desarrollo Tecnológico e Innovación,"229.017,00 €","171.762,75 €"
3,3M ESPAÑA SL,B-28078020,EMPRESA,N,DESARROLLO DE NUEVAS SOLUCIONES DE ECOESTRUCTU...,23/07/2015,Comunidad de Madrid,MADRID,RIVAS-VACIAMADRID,28529,Ayudas Parcialmente Reembolsables,Proyectos Estratégicos de Cooperación (CIEN),Vehículos de transporte,Industria manufacturera,Centro de Desarrollo Tecnológico e Innovación,"716.929,00 €","537.696,75 €"
4,3M ESPAÑA SL,B-28078020,EMPRESA,N,NUEVA TECNOLOGÍA FLEXIBLE DE FABRICACIÓN DE PR...,05/03/2015,Comunidad de Madrid,MADRID,RIVAS-VACIAMADRID,28529,Ayudas Parcialmente Reembolsables,Proyectos CDTI de I+D,Otros sectores industriales (SECTORES INDUSTRI...,Industria manufacturera,Programa Operativo Plurirregional de España,"368.333,00 €","313.083,05 €"


In [4]:
df_cdti.info()

<class 'pandas.DataFrame'>
RangeIndex: 18016 entries, 0 to 18015
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   Razón Social                   18016 non-null  str  
 1   NIF Razón Social               18016 non-null  str  
 2   Tipo Entidad                   18016 non-null  str  
 3   PYME                           17756 non-null  str  
 4   Título Proyecto                18016 non-null  str  
 5   Fecha Aprobación / Resolución  18016 non-null  str  
 6   CCAA                           18016 non-null  str  
 7   Provincia                      18016 non-null  str  
 8   Localidad                      18016 non-null  str  
 9   Código Postal                  18016 non-null  str  
 10  Tipo de Ayuda                  18016 non-null  str  
 11  Instrumento Financiero         17768 non-null  str  
 12  Área Sectorial                 17839 non-null  str  
 13  CNAE                       

## 2. Limpieza de datos
Convertimos las columnas `Presupuesto` y `Aportación CDTI` a `float`

In [5]:
columnas = ['Presupuesto','Aportación CDTI']

for col in columnas:
    df_cdti[col] = df_cdti[col].str.replace('€','').str.replace(' ','').str.replace('.','').str.replace(',','.')

df_cdti = df_cdti.astype({'Presupuesto':float,'Aportación CDTI':float})

df_cdti[['Presupuesto','Aportación CDTI']].describe()

,Presupuesto,Aportación CDTI
count,"18,016.00","18,016.00"
mean,"714,030.44","515,535.65"
std,"1,246,147.52","839,317.38"
min,0.00,0.00
25%,"327,510.25","227,228.70"
50%,"470,359.50","335,000.00"
75%,"749,703.75","551,397.72"
max,"80,521,957.00","36,341,035.65"


Filtramos las filas con presupuesto de 0 € para entender de qué se trata

In [6]:
presu_cero = df_cdti[df_cdti['Presupuesto']==0]

print(presu_cero['Instrumento Financiero'].value_counts())
print(presu_cero['Tipo de Ayuda'].value_counts())

Instrumento Financiero
Innterconecta                           39
CDTI-Eurostars                          10
Misiones MRR                             8
Misiones Ciencia e Innovación            2
Plan Tecnológico Aeronáutico (PTA)       2
TransMisiones                            2
Programa Tecnológico Espacial            2
Ecosistemas / Planes Complementarios     2
CDTI-Eranets                             1
Name: count, dtype: int64
Tipo de Ayuda
Subvención    70
Name: count, dtype: int64


Vemos que los proyectos con presupuesto 0€ son de diferentes convocatorias, por lo que puede deberse a diversos motivos (errores en los datos publicados por CDTI, a renuncias de los beneficiarios, participación en consorcios, etc.).

Realizamos un filtrado similar para analizar si existen proyectos en los que la aportación de CDTI sea 0€.

In [7]:
cdti_cero = df_cdti[df_cdti['Aportación CDTI']==0]

print(cdti_cero['Instrumento Financiero'].value_counts())
print(cdti_cero['Tipo de Ayuda'].value_counts())

Instrumento Financiero
Innterconecta                           39
CDTI-Eurostars                          10
Misiones MRR                             8
Misiones Ciencia e Innovación            2
Plan Tecnológico Aeronáutico (PTA)       2
TransMisiones                            2
Programa Tecnológico Espacial            2
Ecosistemas / Planes Complementarios     2
CDTI-Eranets                             1
Name: count, dtype: int64
Tipo de Ayuda
Subvención    70
Name: count, dtype: int64


Ahora confirmamos que efectivamente se trata de los mismos 70 proyectos en ambos casos, realizando un filtrado de ambas condiciones.

In [8]:
presu_aport_0 = df_cdti[(df_cdti['Presupuesto']==0)&(df_cdti['Aportación CDTI']==0)]
len(presu_aport_0)

70

El filtro con ambas condiciones nos muestra la misma cantidad de proyectos, 70. Esto nos demuestra que son los mismos proyectos los que tienen un valor `0` en `Prespuesto` y en `Aportación CDTI`. Todavía no podemos valorar si se trata de errores o de alguna otra casuística, pero convertimos estos 0€ de ambas columnas en valores NaN para que no distorsionen los cálculos de medias y sumas de importes.

In [9]:
for col in columnas:
    df_cdti[col] = df_cdti[col].replace(0,np.nan)

Comprobamos que todos los `0` se han convertido en `NaN`

In [10]:
df_cdti['Presupuesto'].isna().sum()

np.int64(70)

In [11]:
df_cdti['Aportación CDTI'].isna().sum()

np.int64(70)

Con el método `describe` comprobamos que el `count` en ambas columnas pasa a ser 17,946 y que se eliminan los `0` de `min`.

In [12]:
df_cdti[['Presupuesto','Aportación CDTI']].describe()

,Presupuesto,Aportación CDTI
count,"17,946.00","17,946.00"
mean,"716,815.58","517,546.55"
std,"1,247,775.86","840,333.75"
min,"28,803.00","17,281.80"
25%,"328,912.25","228,561.42"
50%,"471,932.50","335,667.84"
75%,"750,490.00","552,618.39"
max,"80,521,957.00","36,341,035.65"


Buscamos el proyecto de mayor presupuesto para determinar si se trata de un error

In [13]:
max_presu = df_cdti['Presupuesto'].idxmax()
df_cdti.loc[max_presu]

Razón Social                                 LABORATORIOS FARMACEUTICOS ROVI, S.A.
NIF Razón Social                                                        A-28041283
Tipo Entidad                                                               EMPRESA
PYME                                                                             N
Título Proyecto                                                       IPCEI - ROVI
Fecha Aprobación / Resolución                                           07/07/2025
CCAA                                                           Comunidad de Madrid
Provincia                                                                   MADRID
Localidad                                                                   MADRID
Código Postal                                                                28037
Tipo de Ayuda                                                           Subvención
Instrumento Financiero                                                 IPCEI Salud
Área

Vemos que se trata de un proyecto IPCEI (Proyecto Importante de Interés Común Europeo) de Salud. Se trata de un proyecto estratégico a nivel europeo con un alto presupuesto, por lo que no es un error.

El siguiente paso es convertir la columna `Fecha Aprobación / Resolución` a tipo fecha.

In [16]:
df_cdti['Fecha Aprobación / Resolución'] = pd.to_datetime(df_cdti['Fecha Aprobación / Resolución'], format='%d/%m/%Y', dayfirst=True)

Una vez modificado el DataType de la columna lo comprobamos y verificamos que las fechas están en el rango correcto (2015-2025)

In [17]:
print(df_cdti.dtypes)
print(df_cdti['Fecha Aprobación / Resolución'].min())
print(df_cdti['Fecha Aprobación / Resolución'].max())

Razón Social                                str
NIF Razón Social                            str
Tipo Entidad                                str
PYME                                        str
Título Proyecto                             str
Fecha Aprobación / Resolución    datetime64[us]
CCAA                                        str
Provincia                                   str
Localidad                                   str
Código Postal                               str
Tipo de Ayuda                               str
Instrumento Financiero                      str
Área Sectorial                              str
CNAE                                        str
Origen de Fondos                            str
Presupuesto                             float64
Aportación CDTI                         float64
dtype: object
2015-01-28 00:00:00
2025-12-26 00:00:00
